# Lab 00: memory estimate and anatomy of a GGUF

Lab: `corso/laboratori/lab-00-01-stima-memoria e lab-00-02-anatomia-gguf`

Run the cells in order (Runtime > Run all). It downloads up to 1 GB and takes a few minutes.


## 1. Setup

Download the official llama.cpp binaries (build `b11046`, the one used in the course) and the lab files.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # threads = available cores (more than the physical cores slows things down)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Download the sample model (785 MiB, Apache 2.0) into the Hugging Face cache.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Memory estimate (lab 00-01)

`stima_memoria.py` applies the formulas of module 00: weights + KV cache + overhead.

In [ ]:
%%bash
cd $LABS/lab-00-01-stima-memoria
python3 stima_memoria.py --preset qwen3-8b --quant q4_k_m --ctx 8192
python3 stima_memoria.py --params 7.6e9 --quant q4_k_m --ctx 8192 --layers 28 --kv-heads 4 --head-dim 128 | tail -8

**Try it:** change `--ctx` to 32768 and see which line grows.

## 3. Anatomy of a GGUF file (lab 00-02)

In [ ]:
!pip -q install gguf

In [ ]:
%%bash
M=$(ls ~/.cache/huggingface/hub/models--ggml-org--Qwen3.5-0.8B-GGUF/snapshots/*/*.gguf)
ls -la "$M" "$(readlink -f "$M")"
gguf-dump --no-tensors "$M" | head -24
cd $LABS/lab-00-02-anatomia-gguf
python3 anatomia_gguf.py "$M"